# 03 — Preprocessing

This notebook prepares Vela photon event data for pulsar period detection.

## Objectives

- load photon event data from the FITS file;
- convert FITS columns to standard numeric types;
- create relative photon arrival times;
- apply basic energy filtering;
- save the processed dataset for later notebooks.

In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits

In [2]:
PROJECT_ROOT = Path.cwd().parent

raw_data_path = PROJECT_ROOT / "data" / "raw" / "vela_photons.fits"
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_path = processed_dir / "vela_photons_filtered.csv"

processed_dir.mkdir(parents=True, exist_ok=True)

raw_data_path

WindowsPath('C:/Users/vi/pulsar-clean/data/raw/vela_photons.fits')

In [3]:
with fits.open(raw_data_path) as hdul:
    events = hdul[1].data

    df = pd.DataFrame({
        "time": np.asarray(events["TIME"], dtype=np.float64),
        "energy": np.asarray(events["ENERGY"], dtype=np.float64)
    })

df.head()

,time,energy
0,7.804732e+08,106.341278
1,7.805337e+08,326.514008
2,7.805415e+08,207.780685
3,7.805529e+08,229.124054
4,7.805640e+08,271.309235


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 540091 entries, 0 to 540090
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   time    540091 non-null  float64
 1   energy  540091 non-null  float64
dtypes: float64(2)
memory usage: 8.2 MB


## Relative time

Photon arrival times are converted to relative time values. This avoids numerical issues caused by large absolute timestamps and makes later timing calculations easier to interpret.

In [5]:
df["time_rel"] = df["time"] - df["time"].min()

df.head()

,time,energy,time_rel
0,7.804732e+08,106.341278,18748.859121
1,7.805337e+08,326.514008,79221.431558
2,7.805415e+08,207.780685,87021.198388
3,7.805529e+08,229.124054,98457.021602
4,7.805640e+08,271.309235,109584.648355


## Energy filtering

A simple energy filter is applied to remove the lowest-energy photon events. This reduces noise before period detection while keeping most of the useful signal.

The threshold is selected as the 25th percentile of the photon energy distribution.

In [6]:
energy_threshold = df["energy"].quantile(0.25)

df_filtered = df[df["energy"] > energy_threshold].copy()

energy_threshold

np.float64(186.6190414428711)

In [7]:
summary = pd.DataFrame({
    "metric": [
        "original_events",
        "filtered_events",
        "removed_events",
        "removed_percent",
        "energy_threshold"
    ],
    "value": [
        len(df),
        len(df_filtered),
        len(df) - len(df_filtered),
        round((len(df) - len(df_filtered)) / len(df) * 100, 2),
        energy_threshold
    ]
})

summary

,metric,value
0,original_events,540091.000000
1,filtered_events,405068.000000
2,removed_events,135023.000000
3,removed_percent,25.000000
4,energy_threshold,186.619041


In [8]:
df_filtered.describe()

,time,energy,time_rel
count,4.050680e+05,405068.000000,4.050680e+05
mean,7.854331e+08,828.732605,4.978664e+06
std,2.465037e+06,2303.656983,2.465037e+06
min,7.804545e+08,186.619293,0.000000e+00
25%,7.830229e+08,278.054977,2.568402e+06
50%,7.861707e+08,435.466095,5.716275e+06
75%,7.874834e+08,807.090790,7.028916e+06
max,7.901001e+08,278458.968750,9.645593e+06


## Save processed data

The filtered dataset is saved as a CSV file so that later notebooks can load the same preprocessed photon events.

In [9]:
df_filtered.to_csv(processed_path, index=False)

processed_path

WindowsPath('C:/Users/vi/pulsar-clean/data/processed/vela_photons_filtered.csv')

In [10]:
check_df = pd.read_csv(processed_path)

check_df.head()

,time,energy,time_rel
0,7.805337e+08,326.514008,79221.431558
1,7.805415e+08,207.780685,87021.198388
2,7.805529e+08,229.124054,98457.021602
3,7.805640e+08,271.309235,109584.648355
4,7.806201e+08,207.057602,165607.811719


In [11]:
check_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 405068 entries, 0 to 405067
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   time      405068 non-null  float64
 1   energy    405068 non-null  float64
 2   time_rel  405068 non-null  float64
dtypes: float64(3)
memory usage: 9.3 MB


## Summary

The preprocessing step converted the FITS photon event data into a clean tabular format, created relative photon arrival times, applied a basic energy filter, and saved the filtered dataset for period detection.

The resulting dataset is ready for pulsar timing analysis.